# Classifying a ticket with System One Model (Jev)

In [ ]:
from typesafe_sdk import Choice, Noul, Score, TypeSafeClient

import os
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file


True

In [5]:
from datetime import datetime
import time

tickets = [
        "I was charged twice this month — two identical $99 charges on the 1st and 2nd.",
        "How do I export my invoice as a PDF?",
        "Our webhook endpoint stopped receiving events at 14:00 UTC today.",
        "Can you add bulk CSV import for contacts? We need it for our migration.",
        "I can't log in — it says my account is suspended but I received no email about it.",
    ]

start_time = time.perf_counter_ns()
for ticket in tickets:
    with TypeSafeClient(model="jev-latest", api_key=os.getenv("TYPESAFE_API_KEY")) as client:  # Reads TYPESAFE_API_KEY.
        response = client.system_one(
            state=ticket,
            questions={
                "criteria": Choice(
                    instructions="Classify each ticket into exactly one of these categories.",
                    criteria={"billing_dispute": "customer contests a charge or invoice total",
                              "billing_inquiry": "customer asks about charges without disputing them",
                              "payment_failure": "a payment attempt was declined or could not be processed",
                            "technical_error": "a product or API is not working as expected",
                            "account_access": "login, password, or permission issues",
                              "feature_request": "customer asks for capability that doesn't exist yet",
                              "other": "doesn't fit the above categories"
                    },
                ),
                "confidence": Score(
                    instructions="What is the confidence of the classification",
                    criteria=[
                        "High",
                        "Medium",
                        "Low"
                    ],
                )
            }
        )
    print(response.choices["criteria"].choice)
    print(response.scores["confidence"].score)

elapsed_ms = (time.perf_counter_ns() - start_time) / 1_000_000
print(f"Duration: {elapsed_ms:.2f} ms")



billing_dispute
0.54
other
0.73
technical_error
0.95
feature_request
0.67
account_access
0.69
Duration: 1824.01 ms
